In [31]:
#runs in emb_env
import pandas as pd
import fasttext.util
import nltk
import sys
import os
import re
import matplotlib.pyplot as plt
import numpy as np
import networkx as nx
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    permutation_test_score,
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    roc_curve, 
    accuracy_score,
    classification_report,
    roc_auc_score,
    confusion_matrix,
)
from sklearn.pipeline import Pipeline  # Changed from imblearn.pipeline
from scipy.stats import bootstrap
os.chdir(r"C:\Users\Giuliano.DESKTOP-NPATJ24\Desktop\ptsd_classifier_new_code")
sys.path.append(os.path.abspath(r"C:\Users\Giuliano.DESKTOP-NPATJ24\Desktop\ptsd_classifier_new_code"))
from src.functions import (bootstrap_ci)
from src.classifier_function import make_pipelines, _get_scores, run_nested_cv, classifier_builder_multi, participant_level_paired_bootstrap, make_pipelines_ngram, compare_best_families, pick_best_family, permutation_test_nested_cv, run_symptom_cluster_classifiers






dataset = pd.read_csv(
    "data/dataset_preprocessed.csv",
    sep=";",
    decimal=".",
    encoding="utf-8"
)

textvariables = [
    "response_negnt",
    "response_neutr",
    "response_trauma"
]

target_col = "condition_pcl"       # whatever your actual binary label column is called
positive_class = "clin_pcl"          # whatever the positive-class label/string value is

_________________________________
Nested CV 
_________________________________

In [32]:
# 1. Run nested CV across all three conditions (as before)
condition_results, complete_dataset, embedding_cols_by_condition, y_data, pipelines_list = classifier_builder_multi(
    dataset=dataset,
    target_col=target_col,
    positive_class=positive_class,
    response_types=("response_trauma", "response_negnt", "response_neutr"),
)

# 2. Best-family-vs-best-family comparison (not pooled)
best_by_condition, pairwise_best = compare_best_families(condition_results, reference="response_trauma")

# 3. Permutation test — restricted to SVM only, since that's trauma's winning family
perm_results = {}
for rt in ("response_trauma", "response_negnt", "response_neutr"):
    fam, _ = pick_best_family(condition_results[rt])
    fam_pipelines = [p for p in pipelines_list if p[0] == fam]
    X_cond = complete_dataset[embedding_cols_by_condition[rt]].to_numpy()

    print(f"\n=== Permutation test: {rt} (best family: {fam}) ===")
    obs, perm_scores, pval = permutation_test_nested_cv(X_cond, y_data, fam_pipelines, n_permutations=500)
    perm_results[rt] = {"observed": obs, "perm_scores": perm_scores, "pvalue": pval, "family": fam}

Complete-case N (valid embeddings across all conditions): 150
Positive class proportion: 0.2733333333333333

CONDITION: response_trauma

--- Outer Fold 1 ---
Overall best inner: ('gbt', 'depth2', 200), inner AUC=0.694
Overall outer test: AUC=0.710, Acc=0.767
  [gbt] best inner cfg=('depth2', 200), outer AUC=0.710
  [logreg] best inner cfg=('class_weight', 0.01), outer AUC=0.767
  [svm] best inner cfg=('no_weight', 0.01), outer AUC=0.818

--- Outer Fold 2 ---
Overall best inner: ('gbt', 'depth2', 100), inner AUC=0.759
Overall outer test: AUC=0.494, Acc=0.700
  [gbt] best inner cfg=('depth2', 100), outer AUC=0.494
  [logreg] best inner cfg=('no_weight', 100.0), outer AUC=0.670
  [svm] best inner cfg=('no_weight', 0.01), outer AUC=0.670

--- Outer Fold 3 ---
Overall best inner: ('logreg', 'class_weight', 100.0), inner AUC=0.681
Overall outer test: AUC=0.716, Acc=0.733
  [gbt] best inner cfg=('depth2', 100), outer AUC=0.670
  [logreg] best inner cfg=('class_weight', 100.0), outer AUC=0.716

KeyboardInterrupt: 

In [33]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

def report_precision_recall(pooled, threshold=0.5, label=""):
    y_true = pooled["y_true"]
    y_score = pooled["y_score"]
    
    # note: SVM's y_score comes from decision_function, not predict_proba,
    # so 0.5 is NOT the right threshold for SVM -- 0 is (decision boundary)
    y_pred = (y_score >= threshold).astype(int)
    
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    
    print(f"{label}: precision={precision:.3f}, recall={recall:.3f}, f1={f1:.3f}, accuracy={acc:.3f}")
    return precision, recall, f1, acc

In [34]:
report_precision_recall(condition_results["response_trauma"]["per_family_pooled"]["svm"], threshold=0.0, label="Trauma (SVM)")
report_precision_recall(condition_results["response_negnt"]["per_family_pooled"]["gbt"], threshold=0.5, label="Negative (GBT)")
report_precision_recall(condition_results["response_neutr"]["per_family_pooled"]["gbt"], threshold=0.5, label="Neutral (GBT)")

Trauma (SVM): precision=0.434, recall=0.561, f1=0.489, accuracy=0.680
Negative (GBT): precision=0.286, recall=0.146, f1=0.194, accuracy=0.667
Neutral (GBT): precision=0.263, recall=0.122, f1=0.167, accuracy=0.667


(0.2631578947368421,
 0.12195121951219512,
 0.16666666666666666,
 0.6666666666666666)

In [3]:
X_text = complete_dataset["response_trauma"].astype(str).to_numpy()

results_ngram = run_nested_cv(
    X_text, y_data, make_pipelines_ngram(),
    n_outer_splits=5, n_inner_splits=5
)


--- Outer Fold 1 ---
Overall best inner: ('ngram', 'class_weight_(1, 2)', 0.1), inner AUC=0.603
Overall outer test: AUC=0.568, Acc=0.700
  [ngram] best inner cfg=('class_weight_(1, 2)', 0.1), outer AUC=0.568

--- Outer Fold 2 ---
Overall best inner: ('ngram', 'class_weight_(1, 2)', 0.01), inner AUC=0.690
Overall outer test: AUC=0.597, Acc=0.533
  [ngram] best inner cfg=('class_weight_(1, 2)', 0.01), outer AUC=0.597

--- Outer Fold 3 ---
Overall best inner: ('ngram', 'no_weight_(1, 2)', 10.0), inner AUC=0.558
Overall outer test: AUC=0.608, Acc=0.733
  [ngram] best inner cfg=('no_weight_(1, 2)', 10.0), outer AUC=0.608

--- Outer Fold 4 ---
Overall best inner: ('ngram', 'class_weight_(1, 1)', 1.0), inner AUC=0.652
Overall outer test: AUC=0.665, Acc=0.733
  [ngram] best inner cfg=('class_weight_(1, 1)', 1.0), outer AUC=0.665

--- Outer Fold 5 ---
Overall best inner: ('ngram', 'class_weight_(1, 1)', 10.0), inner AUC=0.637
Overall outer test: AUC=0.683, Acc=0.700
  [ngram] best inner cfg=('

In [ ]:
#save the run
import pickle


all_results = {
    "condition_results": condition_results,
    "best_by_condition": best_by_condition,
    "pairwise_best": pairwise_best,
    "perm_results": perm_results,
    "ngram_results": ngram_scores,
}

with open("run_results_gbert_full.pkl", "wb") as f:
    pickle.dump(all_results, f)

In [26]:
#to load the run
import pickle

with open("run_results_gbert_full.pkl", "rb") as f:
    all_results = pickle.load(f)

condition_results = all_results["condition_results"]
ngram_scores = all_results["ngram_results"]

print(type(ngram_scores))
print(ngram_scores)

<class 'numpy.ndarray'>
[0.56818182 0.59659091 0.60795455 0.66477273 0.68253968]


In [20]:
fam, embedding_scores = pick_best_family(condition_results["response_trauma"])

diff = embedding_scores - ngram_scores
diff_low, diff_high = bootstrap_ci(diff)
print(f"Embedding ({fam}) vs n-gram, mean diff: {diff.mean():.3f}, 95% CI: [{diff_low:.3f}, {diff_high:.3f}]")

Embedding (svm) vs n-gram, mean diff: 0.120, 95% CI: [0.070, 0.202]


check for participant level comparison between the narratives and ngrams

In [3]:
condition_results, complete_dataset, embedding_cols_by_condition, y_data, pipelines_list = classifier_builder_multi(
    dataset=dataset,
    target_col=target_col,
    positive_class=positive_class,
    response_types=("response_trauma", "response_negnt", "response_neutr"),
)
X_text_trauma = complete_dataset["response_trauma"].astype(str).to_numpy()

ngram_results = run_nested_cv(
    X_text_trauma, y_data, make_pipelines_ngram(),
    n_outer_splits=5, n_inner_splits=5,
    outer_random_state=0, inner_random_state=1,
)
trauma_svm = condition_results["response_trauma"]["per_family_pooled"]["svm"]
ngram = ngram_results["per_family_pooled"]["ngram"]

assert np.array_equal(trauma_svm["participant_idx"], ngram["participant_idx"])  # confirms alignment

mean_diff, ci_low, ci_high, diffs = participant_level_paired_bootstrap(
    trauma_svm["y_true"], trauma_svm["y_score"], ngram["y_score"]
)
print(f"Participant-level embedding vs n-gram diff: {mean_diff:.3f}, 95% CI [{ci_low:.3f}, {ci_high:.3f}]")

Complete-case N (valid embeddings across all conditions): 150
Positive class proportion: 0.2733333333333333

CONDITION: response_trauma

--- Outer Fold 1 ---
Overall best inner: ('gbt', 'depth2', 200), inner AUC=0.694
Overall outer test: AUC=0.710, Acc=0.767
  [gbt] best inner cfg=('depth2', 200), outer AUC=0.710
  [logreg] best inner cfg=('class_weight', 0.01), outer AUC=0.767
  [svm] best inner cfg=('no_weight', 0.01), outer AUC=0.818

--- Outer Fold 2 ---
Overall best inner: ('gbt', 'depth2', 100), inner AUC=0.759
Overall outer test: AUC=0.494, Acc=0.700
  [gbt] best inner cfg=('depth2', 100), outer AUC=0.494
  [logreg] best inner cfg=('no_weight', 100.0), outer AUC=0.670
  [svm] best inner cfg=('no_weight', 0.01), outer AUC=0.670

--- Outer Fold 3 ---
Overall best inner: ('logreg', 'class_weight', 100.0), inner AUC=0.681
Overall outer test: AUC=0.716, Acc=0.733
  [gbt] best inner cfg=('depth2', 100), outer AUC=0.670
  [logreg] best inner cfg=('class_weight', 100.0), outer AUC=0.716

In [4]:
trauma_svm = condition_results["response_trauma"]["per_family_pooled"]["svm"]
negnt_gbt = condition_results["response_negnt"]["per_family_pooled"]["gbt"]
neutr_gbt = condition_results["response_neutr"]["per_family_pooled"]["gbt"]

assert np.array_equal(trauma_svm["participant_idx"], negnt_gbt["participant_idx"])
assert np.array_equal(trauma_svm["participant_idx"], neutr_gbt["participant_idx"])

mean_diff_negnt, low_negnt, high_negnt, _ = participant_level_paired_bootstrap(
    trauma_svm["y_true"], trauma_svm["y_score"], negnt_gbt["y_score"]
)
print(f"Trauma vs negative, participant-level diff: {mean_diff_negnt:.3f}, 95% CI [{low_negnt:.3f}, {high_negnt:.3f}]")

mean_diff_neutr, low_neutr, high_neutr, _ = participant_level_paired_bootstrap(
    trauma_svm["y_true"], trauma_svm["y_score"], neutr_gbt["y_score"]
)
print(f"Trauma vs neutral, participant-level diff: {mean_diff_neutr:.3f}, 95% CI [{low_neutr:.3f}, {high_neutr:.3f}]")

Trauma vs negative, participant-level diff: 0.159, 95% CI [0.035, 0.280]
Trauma vs neutral, participant-level diff: 0.233, 95% CI [0.103, 0.362]


check der Korrelation von DASS und PCL

In [2]:
from scipy.stats import pearsonr, spearmanr

def report_correlation(df, col_a, col_b, method="pearson"):
    sub = df[[col_a, col_b]].dropna()
    if method == "pearson":
        r, p = pearsonr(sub[col_a], sub[col_b])
    else:
        r, p = spearmanr(sub[col_a], sub[col_b])
    print(f"{col_a} vs {col_b}: r = {r:.3f}, p = {p:.4f}, n = {len(sub)}")
    return r, p

pcl5_col = "PCL-5_total"

dass_cols = {
    "DASS-21 total":      "dass21_total",
    "DASS-21 depression": "DASS21_depression",
    "DASS-21 anxiety":    "DASS21_anxiety",
    "DASS-21 stress":     "DASS21_stress",
}

correlations_spearman = {}
for label, col in dass_cols.items():
    correlations_spearman[label] = report_correlation(dataset, pcl5_col, col, method="spearman")

PCL-5_total vs dass21_total: r = 0.751, p = 0.0000, n = 150
PCL-5_total vs DASS21_depression: r = 0.702, p = 0.0000, n = 150
PCL-5_total vs DASS21_anxiety: r = 0.638, p = 0.0000, n = 150
PCL-5_total vs DASS21_stress: r = 0.713, p = 0.0000, n = 150


Classifiers to predict the DASS21 values

In [3]:
def run_dass_subscale_classifiers(dataset, response_type, dass_targets,
                                   n_outer_splits=5, n_inner_splits=5,
                                   outer_random_state=0, inner_random_state=1):
    """
    dass_targets: dict like {
        "DASS21_depression": ("depression_group", "high"),
        "DASS21_anxiety": ("anxiety_group", "high"),
        "DASS21_stress": ("stress_group", "high"),
    }
    mapping subscale name -> (binary target column, positive class label)
    Assumes the binary group columns already exist in `dataset`,
    OR built from a cutoff -- adjust depending on your answer above.
    """
    embedding_cols = [c for c in dataset.columns if c.startswith(response_type) and "_embedding_" in c]
    pipelines_list = make_pipelines()

    dass_results = {}
    perm_results = {}

    for subscale, (target_col, positive_class) in dass_targets.items():
        print(f"\n{'='*60}\nDASS SUBSCALE: {subscale}\n{'='*60}")

        sub_dataset = dataset.dropna(subset=embedding_cols + [target_col]).reset_index(drop=True)
        X_data = sub_dataset[embedding_cols].to_numpy()
        y_data_sub = ((sub_dataset[target_col] == positive_class) * 1).to_numpy()

        print(f"N = {len(y_data_sub)}, positive proportion = {y_data_sub.mean():.3f}")

        results = run_nested_cv(
            X_data, y_data_sub, pipelines_list,
            n_outer_splits=n_outer_splits, n_inner_splits=n_inner_splits,
            outer_random_state=outer_random_state, inner_random_state=inner_random_state,
        )
        dass_results[subscale] = results

        fam, _ = pick_best_family(results)
        fam_pipelines = [p for p in pipelines_list if p[0] == fam]

        print(f"\n--- Permutation test: {subscale} (best family: {fam}) ---")
        obs, perm_scores, pval = permutation_test_nested_cv(
            X_data, y_data_sub, fam_pipelines, n_permutations=500
        )
        perm_results[subscale] = {"observed": obs, "perm_scores": perm_scores, "pvalue": pval, "family": fam}

    return dass_results, perm_results

In [5]:
dass_targets = {
    "DASS21_depression": ("condition_dass_depression", "clin_dass_depression"),   # adjust column/label to match your actual binarization
    "DASS21_anxiety": ("condition_dass_anxiety", "clin_dass_anxiety"),
    "DASS21_stress": ("condition_dass_stress", "clin_dass_stress"),
}

dass_results, dass_perm_results = run_dass_subscale_classifiers( #run on only the trauma data
    dataset, "response_trauma", dass_targets
)


DASS SUBSCALE: DASS21_depression
N = 150, positive proportion = 0.340

--- Outer Fold 1 ---
Overall best inner: ('logreg', 'class_weight', 0.01), inner AUC=0.409
Overall outer test: AUC=0.545, Acc=0.533
  [gbt] best inner cfg=('depth2', 100), outer AUC=0.632
  [logreg] best inner cfg=('class_weight', 0.01), outer AUC=0.545
  [svm] best inner cfg=('no_weight', 0.01), outer AUC=0.589

--- Outer Fold 2 ---
Overall best inner: ('svm', 'no_weight', 0.1), inner AUC=0.482
Overall outer test: AUC=0.445, Acc=0.533
  [gbt] best inner cfg=('depth3', 100), outer AUC=0.655
  [logreg] best inner cfg=('no_weight', 10.0), outer AUC=0.430
  [svm] best inner cfg=('no_weight', 0.1), outer AUC=0.445

--- Outer Fold 3 ---
Overall best inner: ('svm', 'no_weight', 1.0), inner AUC=0.519
Overall outer test: AUC=0.465, Acc=0.533
  [gbt] best inner cfg=('depth2', 200), outer AUC=0.405
  [logreg] best inner cfg=('no_weight', 10.0), outer AUC=0.445
  [svm] best inner cfg=('no_weight', 1.0), outer AUC=0.465

--- O

In [ ]:
#save the run
import pickle
dass_all_results = {
    "dass_results": dass_results,
    "dass_perm_results": dass_perm_results,
    "correlations_spearman": correlations_spearman,
}

with open("dass_subscale_results.pkl", "wb") as f:
    pickle.dump(dass_all_results, f)

print("Saved.")

Saved.


In [37]:
#load the run
import pickle
with open("dass_subscale_results.pkl", "rb") as f:
    dass_all_results = pickle.load(f)

dass_results = dass_all_results["dass_results"]
dass_perm_results = dass_all_results["dass_perm_results"]
correlations_spearman = dass_all_results["correlations_spearman"]

Check für Classifier-Performance mit DSM-5 PTSD Clusters

In [42]:
# ---- Compute DSM-5 cluster sum scores ----
cluster_items = {
    "Intrusion":            [f"PCL-5_{i}" for i in range(1, 6)],      # items 1-5
    "Avoidance":            [f"PCL-5_{i}" for i in range(6, 8)],      # items 6-7
    "NegCognitionsMood":    [f"PCL-5_{i}" for i in range(8, 15)],     # items 8-14
    "ArousalReactivity":    [f"PCL-5_{i}" for i in range(15, 21)],    # items 15-20
}

for cluster_name, items in cluster_items.items():
    dataset[f"PCL5_{cluster_name}"] = dataset[items].sum(axis=1)

# ---- Median split binarization (no standard clinical cutoff exists per cluster) ----
for cluster_name in cluster_items:
    col = f"PCL5_{cluster_name}"
    median_val = dataset[col].median()
    dataset.loc[dataset[col] > median_val, f"condition_{cluster_name}"] = f"high_{cluster_name}"
    dataset.loc[dataset[col] <= median_val, f"condition_{cluster_name}"] = f"low_{cluster_name}"
    print(f"{cluster_name}: median = {median_val}, split counts:")
    print(dataset[f"condition_{cluster_name}"].value_counts())
    print()

Intrusion: median = 4.0, split counts:
condition_Intrusion
low_Intrusion     78
high_Intrusion    72
Name: count, dtype: int64

Avoidance: median = 2.0, split counts:
condition_Avoidance
low_Avoidance     83
high_Avoidance    67
Name: count, dtype: int64

NegCognitionsMood: median = 7.0, split counts:
condition_NegCognitionsMood
low_NegCognitionsMood     81
high_NegCognitionsMood    69
Name: count, dtype: int64

ArousalReactivity: median = 5.0, split counts:
condition_ArousalReactivity
low_ArousalReactivity     79
high_ArousalReactivity    71
Name: count, dtype: int64



In [43]:
cluster_targets = {
    "Intrusion":         ("condition_Intrusion", "high_Intrusion"),
    "Avoidance":         ("condition_Avoidance", "high_Avoidance"),
    "NegCognitionsMood": ("condition_NegCognitionsMood", "high_NegCognitionsMood"),
    "ArousalReactivity": ("condition_ArousalReactivity", "high_ArousalReactivity"),
}

cluster_results, cluster_perm_results = run_symptom_cluster_classifiers(
    dataset, "response_trauma", cluster_targets
)


DSM-5 CLUSTER: Intrusion
N = 150, positive proportion = 0.480

--- Outer Fold 1 ---


KeyboardInterrupt: 

Testing if AUC is stable over different PCL-5 cutoff values

In [4]:
def run_cutoff_sensitivity(dataset, response_type, pcl_col="PCL-5_total",
                            cutoffs=(31, 33, 34, 36, 38),
                            n_outer_splits=5, n_inner_splits=5,
                            outer_random_state=0, inner_random_state=1):
    embedding_cols = [c for c in dataset.columns if c.startswith(response_type) and "_embedding_" in c]
    pipelines_list = make_pipelines()

    cutoff_results = {}

    for cutoff in cutoffs:
        print(f"\n{'='*60}\nCUTOFF: {cutoff}\n{'='*60}")

        sub_dataset = dataset.dropna(subset=embedding_cols + [pcl_col]).reset_index(drop=True)
        X_data = sub_dataset[embedding_cols].to_numpy()
        y_data_sub = (sub_dataset[pcl_col] >= cutoff).astype(int).to_numpy()

        print(f"N = {len(y_data_sub)}, positive proportion = {y_data_sub.mean():.3f}")

        results = run_nested_cv(
            X_data, y_data_sub, pipelines_list,
            n_outer_splits=n_outer_splits, n_inner_splits=n_inner_splits,
            outer_random_state=outer_random_state, inner_random_state=inner_random_state,
            verbose=False,   # keep this one quiet, we only need the summary
        )
        cutoff_results[cutoff] = results

        fam, scores = pick_best_family(results)
        print(f"Best family: {fam}, mean outer AUC = {scores.mean():.3f} ± {scores.std():.3f}, "
              f"95% CI [{results['auc_ci'][0]:.3f}, {results['auc_ci'][1]:.3f}]")

    return cutoff_results

In [5]:
cutoff_results = run_cutoff_sensitivity(dataset, "response_trauma", cutoffs=(31, 33, 34, 36, 38))


CUTOFF: 31
N = 150, positive proportion = 0.300
Best family: svm, mean outer AUC = 0.727 ± 0.043, 95% CI [0.665, 0.713]

CUTOFF: 33
N = 150, positive proportion = 0.287
Best family: svm, mean outer AUC = 0.678 ± 0.073, 95% CI [0.598, 0.726]

CUTOFF: 34
N = 150, positive proportion = 0.273
Best family: svm, mean outer AUC = 0.744 ± 0.055, 95% CI [0.543, 0.725]

CUTOFF: 36
N = 150, positive proportion = 0.247
Best family: svm, mean outer AUC = 0.684 ± 0.088, 95% CI [0.625, 0.755]

CUTOFF: 38
N = 150, positive proportion = 0.247
Best family: svm, mean outer AUC = 0.684 ± 0.088, 95% CI [0.625, 0.755]


In [13]:
for cutoff in (31, 33, 34, 36, 38):
    fam, scores = pick_best_family(cutoff_results[cutoff])   # <- index by cutoff, not the whole dict
    ci_low, ci_high = bootstrap_ci(scores)
    print(f"Cutoff {cutoff}: family={fam}, mean outer AUC = {scores.mean():.3f} ± {scores.std():.3f}, "
          f"95% CI [{ci_low:.3f}, {ci_high:.3f}]")

Cutoff 31: family=svm, mean outer AUC = 0.727 ± 0.043, 95% CI [0.680, 0.754]
Cutoff 33: family=svm, mean outer AUC = 0.678 ± 0.073, 95% CI [0.602, 0.732]
Cutoff 34: family=svm, mean outer AUC = 0.744 ± 0.055, 95% CI [0.699, 0.794]
Cutoff 36: family=svm, mean outer AUC = 0.684 ± 0.088, 95% CI [0.610, 0.764]
Cutoff 38: family=svm, mean outer AUC = 0.684 ± 0.088, 95% CI [0.610, 0.764]


Analyzing use of first person pronouns

In [5]:
import spacy
import pandas as pd
import numpy as np

nlp = spacy.load("de_core_news_lg")

In [6]:
FIRST_PERSON_FORMS = {
    "ich", "mich", "mir",           # 1st sg: nom, acc, dat
    "wir", "uns",                    # 1st pl: nom, acc/dat
    "mein", "meine", "meiner", "meinem", "meinen", "meines",   # possessive sg
    "unser", "unsere", "unserer", "unserem", "unseren", "unseres",  # possessive pl
}

def extract_first_person_features(text):
    doc = nlp(text)

    n_tokens = len([t for t in doc if not t.is_punct and not t.is_space])
    n_sentences = len(list(doc.sents))

    fp_subject_count = 0      # first-person pronoun as nsubj (grammatical subject)
    fp_nonsubject_count = 0   # first-person pronoun in any other role
    fp_possessive_count = 0   # first-person possessive determiner

    for token in doc:
        lemma_or_text = token.text.lower()
        if lemma_or_text in FIRST_PERSON_FORMS:
            if token.dep_ in ("sb", "nsubj"):       # spaCy German models often use "sb" (subject)
                fp_subject_count += 1
            elif token.pos_ in ("DET",) or token.dep_ in ("ag", "nk") and lemma_or_text.startswith(("mein", "unser")):
                fp_possessive_count += 1
            else:
                fp_nonsubject_count += 1

    return {
        "n_tokens": n_tokens,
        "n_sentences": n_sentences,
        "fp_subject_count": fp_subject_count,
        "fp_nonsubject_count": fp_nonsubject_count,
        "fp_possessive_count": fp_possessive_count,
        "fp_subject_rate": fp_subject_count / n_tokens if n_tokens > 0 else np.nan,
    }


def add_first_person_features(dataset, textvariables):
    for column_name in textvariables:
        for i in range(len(dataset)):
            text = str(dataset.at[i, column_name]).strip()
            if len(text) == 0:
                feats = {"n_tokens": np.nan, "n_sentences": np.nan,
                          "fp_subject_count": np.nan, "fp_nonsubject_count": np.nan,
                          "fp_possessive_count": np.nan, "fp_subject_rate": np.nan}
            else:
                feats = extract_first_person_features(text)
            for key, val in feats.items():
                dataset.loc[i, f"{column_name}_{key}"] = val
    return dataset

In [7]:
dataset = add_first_person_features(dataset, textvariables)

In [9]:
from scipy.stats import ttest_ind

high_group = dataset.loc[dataset[target_col] == positive_class, "response_trauma_fp_subject_rate"].dropna()
low_group = dataset.loc[dataset[target_col] != positive_class, "response_trauma_fp_subject_rate"].dropna()

t_stat, p_val = ttest_ind(high_group, low_group, equal_var=False)  # Welch's t-test, matching LIWC's approach
print(f"First-person subject rate, high vs low PCL-5 group: t = {t_stat:.2f}, p = {p_val:.3f}, "
      f"n_high = {len(high_group)}, n_low = {len(low_group)}")

First-person subject rate, high vs low PCL-5 group: t = 0.95, p = 0.344, n_high = 41, n_low = 109


In [24]:
print(dataset[[c for c in dataset.columns if c.endswith("_wc")]].max())

response_negnt_wc     311.0
response_neutr_wc     208.0
response_trauma_wc    318.0
dtype: float64


Classifier using the subjective measures of the participants as done by Giuliani et. al.

In [10]:
def run_subjective_classifier(dataset, response_type, target_col, positive_class,
                               n_outer_splits=5, n_inner_splits=5,
                               outer_random_state=0, inner_random_state=1):

    feature_cols = [
        f"vivid_{response_type.split('_')[1]}_1",
        f"emoArous_{response_type.split('_')[1]}_1",
        f"valence_{response_type.split('_')[1]}_1",
        f"detail_{response_type.split('_')[1]}_1",
        f"relevance_{response_type.split('_')[1]}_1",
        f"imagery_{response_type.split('_')[1]}_1",
        f"perspec_{response_type.split('_')[1]}_1",
        f"active_{response_type.split('_')[1]}_1",
        f"ease_{response_type.split('_')[1]}_1",
    ]

    sub_dataset = dataset.dropna(subset=feature_cols + [target_col]).reset_index(drop=True)
    X_data = sub_dataset[feature_cols].to_numpy()
    y_data_sub = ((sub_dataset[target_col] == positive_class) * 1).to_numpy()

    print(f"N = {len(y_data_sub)}, positive proportion = {y_data_sub.mean():.3f}")

    pipelines_list = make_pipelines()

    results = run_nested_cv(
        X_data, y_data_sub, pipelines_list,
        n_outer_splits=n_outer_splits, n_inner_splits=n_inner_splits,
        outer_random_state=outer_random_state, inner_random_state=inner_random_state,
    )

    fam, _ = pick_best_family(results)
    fam_pipelines = [p for p in pipelines_list if p[0] == fam]

    print(f"\n--- Permutation test: subjective measures (best family: {fam}) ---")
    obs, perm_scores, pval = permutation_test_nested_cv(
        X_data, y_data_sub, fam_pipelines, n_permutations=500
    )

    return results, {"observed": obs, "perm_scores": perm_scores, "pvalue": pval, "family": fam}

In [11]:
subjective_results, subjective_perm = run_subjective_classifier(
    dataset, "response_trauma", "condition_pcl", "clin_pcl"
)

N = 150, positive proportion = 0.273

--- Outer Fold 1 ---
Overall best inner: ('logreg', 'no_weight', 0.01), inner AUC=0.662
Overall outer test: AUC=0.443, Acc=0.600
  [gbt] best inner cfg=('depth3', 200), outer AUC=0.585
  [logreg] best inner cfg=('no_weight', 0.01), outer AUC=0.443
  [svm] best inner cfg=('no_weight', 0.01), outer AUC=0.545

--- Outer Fold 2 ---
Overall best inner: ('svm', 'class_weight', 0.01), inner AUC=0.599
Overall outer test: AUC=0.716, Acc=0.633
  [gbt] best inner cfg=('depth3', 200), outer AUC=0.699
  [logreg] best inner cfg=('class_weight', 0.1), outer AUC=0.722
  [svm] best inner cfg=('class_weight', 0.01), outer AUC=0.716

--- Outer Fold 3 ---
Overall best inner: ('logreg', 'class_weight', 0.01), inner AUC=0.576
Overall outer test: AUC=0.435, Acc=0.400
  [gbt] best inner cfg=('depth2', 100), outer AUC=0.293
  [logreg] best inner cfg=('class_weight', 0.01), outer AUC=0.435
  [svm] best inner cfg=('no_weight', 0.01), outer AUC=0.389

--- Outer Fold 4 ---
Ove

In [28]:
trauma_fam, trauma_scores = pick_best_family(
    condition_results["response_trauma"]
)
subj_fam, subj_scores = pick_best_family(subjective_results)
subjective_pooled = subjective_results["per_family_pooled"][subj_fam]

assert np.array_equal(trauma_svm["participant_idx"], subjective_pooled["participant_idx"])

mean_diff, ci_low, ci_high, diffs = participant_level_paired_bootstrap(
    trauma_svm["y_true"], trauma_svm["y_score"], subjective_pooled["y_score"]
)
print(f"Embedding vs subjective measures diff: {mean_diff:.3f}, 95% CI [{ci_low:.3f}, {ci_high:.3f}]")

Embedding vs subjective measures diff: 0.123, 95% CI [-0.025, 0.264]


All metrics of all nested cvs combined (no permutation tests)

In [44]:
all_metrics = {}

# ---------- 1. Trauma / negative / neutral (classifier_builder_multi has no permutation calls itself) ----------
condition_results, complete_dataset, embedding_cols_by_condition, y_data, pipelines_list = classifier_builder_multi(
    dataset=dataset,
    target_col=target_col,
    positive_class=positive_class,
    response_types=("response_trauma", "response_negnt", "response_neutr"),
)
for rt, label in [("response_trauma", "Trauma"), ("response_negnt", "Negative"), ("response_neutr", "Neutral")]:
    fam, precision, recall, f1, acc = report_best_family_metrics(condition_results[rt], label)
    all_metrics[label] = (fam, precision, recall, f1, acc)

# ---------- 2. N-gram baseline ----------
X_text_trauma = complete_dataset["response_trauma"].astype(str).to_numpy()
ngram_results = run_nested_cv(X_text_trauma, y_data, make_pipelines_ngram(), n_outer_splits=5, n_inner_splits=5)
fam, precision, recall, f1, acc = report_best_family_metrics(ngram_results, "N-gram baseline")
all_metrics["N-gram baseline"] = (fam, precision, recall, f1, acc)

# ---------- 3. DASS-21 subscales ----------
dass_targets = {
    "DASS21_depression": ("condition_dass_depression", "clin_dass_depression"),
    "DASS21_anxiety": ("condition_dass_anxiety", "clin_dass_anxiety"),
    "DASS21_stress": ("condition_dass_stress", "clin_dass_stress"),
}
embedding_cols_trauma = [c for c in dataset.columns if c.startswith("response_trauma") and "_embedding_" in c]
dass_results_metrics = {}
for subscale, (tcol, pclass) in dass_targets.items():
    sub_dataset = dataset.dropna(subset=embedding_cols_trauma + [tcol]).reset_index(drop=True)
    X_data = sub_dataset[embedding_cols_trauma].to_numpy()
    y_data_sub = ((sub_dataset[tcol] == pclass) * 1).to_numpy()
    results = run_nested_cv(X_data, y_data_sub, make_pipelines(), n_outer_splits=5, n_inner_splits=5)
    dass_results_metrics[subscale] = results
    fam, precision, recall, f1, acc = report_best_family_metrics(results, subscale)
    all_metrics[subscale] = (fam, precision, recall, f1, acc)

# ---------- 4. DSM-5 symptom clusters ----------
cluster_targets = {
    "Intrusion": ("condition_Intrusion", "high_Intrusion"),
    "Avoidance": ("condition_Avoidance", "high_Avoidance"),
    "NegCognitionsMood": ("condition_NegCognitionsMood", "high_NegCognitionsMood"),
    "ArousalReactivity": ("condition_ArousalReactivity", "high_ArousalReactivity"),
}
cluster_results_metrics = {}
for cluster, (tcol, pclass) in cluster_targets.items():
    sub_dataset = dataset.dropna(subset=embedding_cols_trauma + [tcol]).reset_index(drop=True)
    X_data = sub_dataset[embedding_cols_trauma].to_numpy()
    y_data_sub = ((sub_dataset[tcol] == pclass) * 1).to_numpy()
    results = run_nested_cv(X_data, y_data_sub, make_pipelines(), n_outer_splits=5, n_inner_splits=5)
    cluster_results_metrics[cluster] = results
    fam, precision, recall, f1, acc = report_best_family_metrics(results, cluster)
    all_metrics[cluster] = (fam, precision, recall, f1, acc)

# ---------- 5. Subjective measures ----------
feature_cols = [f"{p}_trauma_1" for p in ["vivid", "emoArous", "valence", "detail", "relevance", "imagery", "perspec", "active", "ease"]]
sub_dataset = dataset.dropna(subset=feature_cols + [target_col]).reset_index(drop=True)
X_data = sub_dataset[feature_cols].to_numpy()
y_data_sub = ((sub_dataset[target_col] == positive_class) * 1).to_numpy()
subjective_results = run_nested_cv(X_data, y_data_sub, make_pipelines(), n_outer_splits=5, n_inner_splits=5)
fam, precision, recall, f1, acc = report_best_family_metrics(subjective_results, "Subjective measures")
all_metrics["Subjective measures"] = (fam, precision, recall, f1, acc)

# ---------- Summary ----------
print("\n=== ALL METRICS ===")
for label, (fam, p, r, f, a) in all_metrics.items():
    print(f"{label} ({fam}): precision={p:.3f}, recall={r:.3f}, f1={f:.3f}, accuracy={a:.3f}")

Complete-case N (valid embeddings across all conditions): 150
Positive class proportion: 0.2733333333333333

CONDITION: response_trauma

--- Outer Fold 1 ---
Overall best inner: ('gbt', 'depth2', 200), inner AUC=0.694
Overall outer test: AUC=0.710, Acc=0.767
  [gbt] best inner cfg=('depth2', 200), outer AUC=0.710
  [logreg] best inner cfg=('class_weight', 0.01), outer AUC=0.767
  [svm] best inner cfg=('no_weight', 0.01), outer AUC=0.818

--- Outer Fold 2 ---
Overall best inner: ('gbt', 'depth2', 100), inner AUC=0.759
Overall outer test: AUC=0.494, Acc=0.700
  [gbt] best inner cfg=('depth2', 100), outer AUC=0.494
  [logreg] best inner cfg=('no_weight', 100.0), outer AUC=0.670
  [svm] best inner cfg=('no_weight', 0.01), outer AUC=0.670

--- Outer Fold 3 ---
Overall best inner: ('logreg', 'class_weight', 100.0), inner AUC=0.681
Overall outer test: AUC=0.716, Acc=0.733
  [gbt] best inner cfg=('depth2', 100), outer AUC=0.670
  [logreg] best inner cfg=('class_weight', 100.0), outer AUC=0.716